# FS13-14 stable unified


In [ ]:
import os, json, math, random, time
from pathlib import Path
import numpy as np
os.environ.pop('CUDA_VISIBLE_DEVICES', None)
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path('/kaggle/working'); FIG=OUT/'figures'; RES=OUT/'results'
FIG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device',device,'gpus',torch.cuda.device_count() if torch.cuda.is_available() else 0)
PROGRESS={}; GATES={}
def gate(name, ok, detail=''):
    GATES[name]=bool(ok); print(('PASS' if ok else 'FAIL'), name, detail)
    if not ok: raise AssertionError(f'ACCEPTANCE FAILED: {name} {detail}')
def make_shape_image(kind, size=64):
    img=np.ones((size,size,3),np.float32)*0.95
    yy,xx=np.mgrid[0:size,0:size]; cy,cx=size//2,size//2
    if kind=='red_circle':
        m=(yy-cy)**2+(xx-cx)**2<=(size*0.28)**2; img[m]=(0.9,0.15,0.12)
    elif kind=='blue_square':
        m=(np.abs(yy-cy)<size*0.25)&(np.abs(xx-cx)<size*0.25); img[m]=(0.15,0.25,0.85)
    elif kind=='green_triangle':
        top=cy-int(size*0.28); bot=cy+int(size*0.30)
        for y in range(max(0,top),min(size,bot)):
            half=int((y-top)/max(1,bot-top)*size*0.30)
            img[y, max(0,cx-half):min(size,cx+half+1)]=(0.15,0.75,0.25)
    else: raise ValueError(kind)
    return img
CLASSES=['red_circle','blue_square','green_triangle']
c2i={c:i for i,c in enumerate(CLASSES)}
CAPTIONS={c:'a '+c.replace('_',' ') for c in CLASSES}


## FS13


In [ ]:
ANS_VOCAB=['<pad>','<bos>','<eos>','red','blue','green','circle','square','triangle','yes','no']
av={t:i for i,t in enumerate(ANS_VOCAB)}
Q_VOCAB=['<pad>','what','color','shape','is','it','a','?']
qv={t:i for i,t in enumerate(Q_VOCAB)}
QA=[
 ('red_circle',['what','color','?'],['red']),
 ('red_circle',['what','shape','?'],['circle']),
 ('blue_square',['what','color','?'],['blue']),
 ('blue_square',['is','it','a','square','?'],['yes']),
 ('green_triangle',['what','shape','?'],['triangle']),
 ('green_triangle',['is','it','a','circle','?'],['no']),
]
class ImgTok(nn.Module):
    def __init__(self,d=64):
        super().__init__()
        self.cnn=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,d,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d((2,2)))
    def forward(self,x): return self.cnn(x).flatten(2).transpose(1,2)
class TinyMLLM(nn.Module):
    def __init__(self,d=64):
        super().__init__()
        self.img=ImgTok(d); self.qemb=nn.Embedding(len(Q_VOCAB),d); self.aemb=nn.Embedding(len(ANS_VOCAB),d)
        layer=nn.TransformerEncoderLayer(d_model=d,nhead=4,dim_feedforward=128,batch_first=True,dropout=0.0)
        self.tr=nn.TransformerEncoder(layer,num_layers=2); self.head=nn.Linear(d,len(ANS_VOCAB))
    def forward(self,images,qids,aids_in):
        seq=torch.cat([self.img(images),self.qemb(qids),self.aemb(aids_in)],1)
        L=seq.size(1); causal=torch.triu(torch.ones(L,L,device=seq.device),1).bool()
        h=self.tr(seq,mask=causal); return self.head(h[:,-aids_in.size(1):,:])
def encode_q(toks,L=6):
    ids=[qv.get(t,0) for t in toks][:L]; return ids+[0]*(L-len(ids))
def encode_a(toks,L=4):
    ids=[av['<bos>']]+[av[t] for t in toks]+[av['<eos>']]
    return (ids+[av['<pad>']]*L)[:L]
def batch(B=48):
    ims,qs,ain,aout=[],[],[],[]
    for _ in range(B):
        k,q,a=random.choice(QA)
        img=np.clip(make_shape_image(k,32)+0.02*np.random.randn(32,32,3).astype(np.float32),0,1)
        ims.append(img.transpose(2,0,1)); qs.append(encode_q(q))
        full=encode_a(a); ain.append(full[:-1]); aout.append(full[1:])
    return torch.tensor(np.stack(ims),dtype=torch.float32),torch.tensor(qs),torch.tensor(ain),torch.tensor(aout)
mllm=TinyMLLM().to(device); opt=torch.optim.Adam(mllm.parameters(),lr=2e-3)
hist13=[]
for epoch in range(1,50):
    mllm.train(); losses=[]; accs=[]
    for _ in range(60):
        im,q,ain,aout=[t.to(device) for t in batch()]
        opt.zero_grad(set_to_none=True)
        lg=mllm(im,q,ain)
        loss=F.cross_entropy(lg.reshape(-1,len(ANS_VOCAB)),aout.reshape(-1),ignore_index=av['<pad>'])
        loss.backward(); opt.step(); losses.append(loss.item())
        mask=aout!=av['<pad>']; accs.append((lg.argmax(-1)[mask]==aout[mask]).float().mean().item())
    hist13.append({'epoch':epoch,'loss':round(float(np.mean(losses)),4),'token_acc':round(float(np.mean(accs)),4)})
    if epoch%10==0: print(hist13[-1])

@torch.no_grad()
def mllm_answer(kind,qtoks):
    mllm.eval()
    im=torch.tensor(make_shape_image(kind,32).transpose(2,0,1)[None],dtype=torch.float32,device=device)
    q=torch.tensor([encode_q(qtoks)],device=device)
    ids=[av['<bos>']]; outs=[]
    for _ in range(3):
        cur=ids+[av['<pad>']]*(3-len(ids)); ain=torch.tensor([cur[:3]],device=device)
        lg=mllm(im,q,ain); nxt=int(lg[0,len(ids)-1].argmax().item())
        if nxt in (av['<eos>'],av['<pad>']): break
        outs.append(ANS_VOCAB[nxt]); ids.append(nxt)
    return ' '.join(outs)
rows13=[]
for k,q,a in QA:
    pred=mllm_answer(k,q)
    rows13.append({'image':k,'q':' '.join(q),'gt':' '.join(a),'pred':pred,'ok':pred.strip()==' '.join(a)})
acc13=sum(r['ok'] for r in rows13)/len(rows13)
print(rows13,acc13)
gate('FS13_acc', acc13>=0.999, rows13)
fig,ax=plt.subplots(figsize=(7,2.8)); ax.axis('off')
ax.table(cellText=[[r['image'],r['q'],r['gt'],r['pred'],str(r['ok'])] for r in rows13],
         colLabels=['image','q','gt','pred','ok'],loc='center',cellLoc='center').set_fontsize(7)
fig.tight_layout(); fig.savefig(FIG/'fs13_mllm.png',dpi=140); plt.close()
(RES/'fs13.json').write_text(json.dumps({'stage':'FS13','method':'tiny MLLM causal','acc':acc13,'rows':rows13,'history':hist13,'vs_prev':'fusion->AR'},indent=2))
PROGRESS['FS13']='ok'


## FS14


In [ ]:
WORD8=['red','blue','green','circle','square','triangle','yes','no']
w8={w:i for i,w in enumerate(WORD8)}
FREQ={'red':180.0,'blue':540.0,'green':1200.0}

class AnyToAny(nn.Module):
    def __init__(self,d=96):
        super().__init__()
        self.img_in=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(64,d))
        self.txt_in=nn.Embedding(8,d)
        self.aud_in=nn.Sequential(nn.Conv1d(1,32,9,stride=2),nn.ReLU(),nn.Conv1d(32,64,9,stride=2),nn.ReLU(),
                                  nn.AdaptiveAvgPool1d(1),nn.Flatten(),nn.Linear(64,d))
        self.trunk=nn.Sequential(nn.Linear(d,d),nn.ReLU())
        self.out_txt=nn.Linear(d,8)
        self.out_img=nn.Sequential(nn.Linear(d,3*16*16),nn.Sigmoid())
    def encode(self,mod,payload):
        if mod=='image': return self.img_in(payload)
        if mod=='text': return self.txt_in(payload).mean(1)
        if mod=='audio': return self.aud_in(payload)
        raise ValueError(mod)
    def forward(self,in_mod,out_mod,payload):
        z=self.trunk(self.encode(in_mod,payload))
        if out_mod=='text': return self.out_txt(z)
        if out_mod=='image': return self.out_img(z).view(-1,3,16,16)
        raise ValueError(out_mod)

router=AnyToAny().to(device)

def img_batch(B=64):
    xs,ys=[],[]
    for _ in range(B):
        k=random.choice(CLASSES)
        img=np.clip(make_shape_image(k,32)+0.02*np.random.randn(32,32,3).astype(np.float32),0,1)
        xs.append(img.transpose(2,0,1)); ys.append(w8[k.split('_')[0]])
    return torch.tensor(np.stack(xs),dtype=torch.float32), torch.tensor(ys)

def aud_batch(B=64,n=2000):
    xs,ys=[],[]
    for _ in range(B):
        color=random.choice(list(FREQ))
        t=np.linspace(0,0.3,n,endpoint=False)
        wav=(0.7*np.sin(2*np.pi*FREQ[color]*t)+0.01*np.random.randn(n)).astype(np.float32)
        xs.append(wav[None]); ys.append(w8[color])
    return torch.tensor(np.stack(xs),dtype=torch.float32), torch.tensor(ys)

def txt_img_batch():
    texts=[]; imgs=[]
    for k in CLASSES:
        texts.append([w8[k.split('_')[0]], w8[k.split('_')[1]]])
        imgs.append(make_shape_image(k,16).transpose(2,0,1))
    # tile
    return torch.tensor(texts*20,dtype=torch.long), torch.tensor(np.stack(imgs*20),dtype=torch.float32)

# Phase training
opt=torch.optim.Adam(router.parameters(), lr=2e-3)
hist14=[]
for phase, steps in [('image', 80),('audio', 80),('txtimg', 60),('joint', 40)]:
    for epoch in range(1, steps+1):
        router.train(); losses={}
        if phase in ('image','joint'):
            xb,yb=img_batch(); xb,yb=xb.to(device),yb.to(device)
            l1=F.cross_entropy(router('image','text',xb),yb)
        else:
            l1=torch.tensor(0.,device=device)
        if phase in ('audio','joint'):
            ab,ay=aud_batch(); ab,ay=ab.to(device),ay.to(device)
            l2=F.cross_entropy(router('audio','text',ab),ay)
        else:
            l2=torch.tensor(0.,device=device)
        if phase in ('txtimg','joint'):
            tb,ib=txt_img_batch(); tb,ib=tb.to(device),ib.to(device)
            l3=F.mse_loss(router('text','image',tb),ib)
        else:
            l3=torch.tensor(0.,device=device)
        loss=l1+l2+l3
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if epoch%20==0 or epoch==steps:
            hist14.append({'phase':phase,'epoch':epoch,'loss':round(float(loss.item()),4),
                           'l1':round(float(l1.item()),4),'l2':round(float(l2.item()),4),'l3':round(float(l3.item()),4)})
            print(hist14[-1])

router.eval(); routes=[]
with torch.no_grad():
    for k in CLASSES:
        # majority over 3 noisy views
        preds=[]
        for _ in range(3):
            img=np.clip(make_shape_image(k,32)+0.01*np.random.randn(32,32,3).astype(np.float32),0,1)
            x=torch.tensor(img.transpose(2,0,1)[None],dtype=torch.float32,device=device)
            preds.append(WORD8[router('image','text',x).argmax(1).item()])
        pred=max(set(preds), key=preds.count)
        routes.append({'route':'image->text','in':k,'out':pred,'ok':pred==k.split('_')[0]})
    for color,f in FREQ.items():
        t=np.linspace(0,0.3,2000,endpoint=False)
        wav=(0.7*np.sin(2*np.pi*f*t)).astype(np.float32)
        a=torch.tensor(wav[None,None],dtype=torch.float32,device=device)
        pred=WORD8[router('audio','text',a).argmax(1).item()]
        routes.append({'route':'audio->text','in':color,'out':pred,'ok':pred==color})
    fig,axes=plt.subplots(1,3,figsize=(7,2.5))
    for j,k in enumerate(CLASSES):
        ids=torch.tensor([[w8[k.split('_')[0]], w8[k.split('_')[1]]]],device=device)
        im=router('text','image',ids)[0].cpu().numpy().transpose(1,2,0)
        axes[j].imshow(np.clip(im,0,1)); axes[j].set_title('text->'+k,fontsize=8); axes[j].axis('off')
        routes.append({'route':'text->image','in':k,'mean_rgb':[round(float(x),3) for x in im.mean((0,1))]})
fig.tight_layout(); fig.savefig(FIG/'fs14_any2any.png',dpi=120); plt.close()
ok=[r for r in routes if 'ok' in r]
acc14=sum(r['ok'] for r in ok)/len(ok)
print(routes, acc14)
gate('FS14_route_acc', acc14>=0.999, routes)
(RES/'fs14.json').write_text(json.dumps({'stage':'FS14','method':'phased any-to-any router','route_acc':acc14,'routes':routes,'history':hist14,'vs_prev':'single interface->multi-route'},indent=2))
PROGRESS['FS14']='ok'
(RES/'FINAL_CURRICULUM.json').write_text(json.dumps({'stages':list(PROGRESS.keys()),'gates':GATES},indent=2))
(OUT/'SUCCESS').write_text('ok\n')
(OUT/'ACCEPTANCE.json').write_text(json.dumps({'ok':all(GATES.values()),'gates':GATES,'progress':PROGRESS},indent=2))
print('FS13-14 PASS',GATES)
